In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit_ibm_runtime import QiskitRuntimeService, Session, Sampler
import timeit

class IBMQuantumGrover:
    def __init__(self, api_token='hc_jCRxZWVqz8KZuZY58kmtbiC81I1tuGuKrU4a-VOGF', channel="ibm_cloud"):
        """
        Initialisiert den IBM Quantum Service über Qiskit Runtime
        
        Args:
            api_token: Ihr persönliches IBM Quantum API-Token
            channel: API-Kanal für IBM Quantum
        """
        self.service = QiskitRuntimeService(channel=channel, token=api_token)
        self.job_ids = {}

    def linear_search(self, items, target):
        """Klassische lineare Suche"""
        return items.index(target) if target in items else -1

    def create_grover_oracle(self, n_qubits, target_state):
        """Erstellt ein Orakel für den Zielzustand |11...1>"""
        qc = QuantumCircuit(n_qubits, name="oracle")
        if target_state == (2**n_qubits - 1):
            qc.x(range(n_qubits))
            qc.h(n_qubits-1)
            qc.mcx(list(range(n_qubits-1)), n_qubits-1)
            qc.h(n_qubits-1)
            qc.x(range(n_qubits))
        else:
            raise NotImplementedError("Orakel nur für |11...1> implementiert")
        return qc

    def run_grover(self, n_qubits, backend_name="ibmq_lima", shots=1024):
        """
        Führt Grover auf IBM Quantum-Hardware aus und gibt Resultat zurück
        """
        target_state = 2 ** n_qubits - 1
        qc = QuantumCircuit(n_qubits, n_qubits)
        qc.h(range(n_qubits))
        iterations = int(np.pi / 4 * np.sqrt(2 ** n_qubits))
        for _ in range(iterations):
            qc.append(self.create_grover_oracle(n_qubits, target_state), range(n_qubits))
            qc.h(range(n_qubits))
            qc.x(range(n_qubits))
            qc.h(n_qubits-1)
            qc.mcx(list(range(n_qubits-1)), n_qubits-1)
            qc.h(n_qubits-1)
            qc.x(range(n_qubits))
            qc.h(range(n_qubits))
        qc.measure(range(n_qubits), range(n_qubits))

        transpiled_qc = transpile(qc, optimization_level=3)
        start_time = time.time()

        # IBM Hardware-Ausführung über Session und Sampler (Qiskit Runtime)
        with Session(service=self.service, backend=backend_name) as session:
            sampler = Sampler(session=session)
            result = sampler.run(transpiled_qc, shots=shots).result()
            counts = result.quasi_dists[0]
            job_id = result.metadata["job_id"] if "job_id" in result.metadata else None

        self.job_ids[job_id] = {
            "n_qubits": n_qubits,
            "backend": backend_name,
            "timestamp": start_time
        }
        return job_id, time.time() - start_time, transpiled_qc, counts

    def compare_algorithms(self, max_qubits=5, backend_name="ibmq_lima"):
        """Vergleich klassische vs. Quanten-Suche mit echter Hardware"""
        sizes = [2 ** n for n in range(1, max_qubits + 1)]
        results = []
        for size in sizes:
            print(f"\nVerarbeite Größe {size}...")
            items = list(range(size))
            target = size - 1
            classical_time = np.mean([
                timeit.timeit(lambda: self.linear_search(items, target), number=100)
                for _ in range(3)
            ])
            n_qubits = int(np.log2(size))
            job_id, quantum_time, _, quantum_counts = self.run_grover(n_qubits, backend_name)
            results.append({
                "size": size,
                "classical_time": classical_time,
                "quantum_time": quantum_time,
                "job_id": job_id,
                "n_qubits": n_qubits,
                "counts": quantum_counts
            })
            print(f"Klassisch: {classical_time:.4f}s | Quanten (Job {job_id}): {quantum_time:.4f}s")
        self.save_results(results)
        self.plot_results(results)
        return results

    def save_results(self, results, filename="quantum_results.json"):
        """Speichert Ergebnisse samt Job-IDs"""
        import json
        with open(filename, 'w') as f:
            json.dump(results, f, indent=2)
        print(f"Ergebnisse gespeichert in {filename}")

    def plot_results(self, results):
        """Visualisiert die gesammelten Ergebnisse"""
        sizes = [r['size'] for r in results]
        classical = [r['classical_time'] for r in results]
        quantum = [r['quantum_time'] for r in results]
        plt.figure(figsize=(10, 6))
        plt.plot(sizes, classical, 'b-o', label='Lineare Suche (O(n))')
        plt.plot(sizes, quantum, 'r--o', label='Grover auf IBMQ (Job-Zeit)')
        plt.xscale('log', base=2)
        plt.yscale('log')
        plt.xlabel('Anzahl der Elemente (log-Skala)')
        plt.ylabel('Zeit (Sekunden, log-Skala)')
        plt.title('Leistungsvergleich: Klassisch vs. Quanten (IBM Hardware)')
        plt.legend()
        plt.grid(True, which="both", ls="--")
        plt.savefig('ibmq_grover_results.png', dpi=300, bbox_inches='tight')
        plt.show()

if __name__ == "__main__":
    API_TOKEN = "hc_jCRxZWVqz8KZuZY58kmtbiC81I1tuGuKrU4a-VOGF"  # Hier deinen IBM Quantum API Token eintragen
    BACKEND = "ibmq_lima"  # Oder Simulator: "ibmq_qasm_simulator"
    grover = IBMQuantumGrover(api_token=API_TOKEN)
    results = grover.compare_algorithms(max_qubits=5, backend_name=BACKEND)
    # Nachverarbeitung und Ergebnisabfrage ist direkt integriert


In [1]:
import sys
print(sys.executable)


c:\Users\juanc\Documents\Ausbildung_Informatik\1_Praktikum\Praktikum_Daten-_und_Prozessanalyse\Python_JNotebook\Qiskit\qiskit_env\Scripts\python.exe


In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt
import timeit
from qiskit import QuantumCircuit, transpile
from qiskit_ibm_runtime import QiskitRuntimeService, Sampler

class IBMQuantumGrover:
    def __init__(self, api_token=None, instance=None, channel="ibm_quantum_platform"):
        # Service-Initialisierung mit API-Token, Kanal und Instanz-CRN
        self.service = QiskitRuntimeService(
            channel=channel,
            token=api_token,
            instance=instance
        )
        self.job_ids = {}

    def linear_search(self, items, target):
        """Klassische lineare Suche."""
        return items.index(target) if target in items else -1

    def create_grover_oracle(self, n_qubits, target_state):
        """Erstellt Orakel für Zielzustand |11...1>."""
        if n_qubits < 2:
            raise ValueError("Grover benötigt mindestens 2 Qubits!")
        qc = QuantumCircuit(n_qubits, name="oracle")
        if target_state == (2**n_qubits - 1):
            qc.x(range(n_qubits))
            qc.h(n_qubits-1)
            qc.mcx(list(range(n_qubits-1)), n_qubits-1)
            qc.h(n_qubits-1)
            qc.x(range(n_qubits))
        else:
            raise NotImplementedError("Orakel nur für |11...1> implementiert")
        return qc

    def run_grover(self, n_qubits, backend_name, shots=1024):
        target_state = 2 ** n_qubits - 1
        qc = QuantumCircuit(n_qubits, n_qubits)
        qc.h(range(n_qubits))
        iterations = int(np.floor(np.pi / 4 * np.sqrt(2 ** n_qubits)))
        for _ in range(iterations):
            qc.append(self.create_grover_oracle(n_qubits, target_state), range(n_qubits))
            qc.h(range(n_qubits))
            qc.x(range(n_qubits))
            qc.h(n_qubits-1)
            qc.mcx(list(range(n_qubits-1)), n_qubits-1)
            qc.h(n_qubits-1)
            qc.x(range(n_qubits))
            qc.h(range(n_qubits))
        qc.measure(range(n_qubits), range(n_qubits))
        transpiled_qc = transpile(qc, optimization_level=3)
        start_time = time.time()

        backend = self.service.backend(backend_name)
        sampler = Sampler()
        # Korrekte aktuelle API: primitive_backend als Parameter
        result = sampler.run(circuits=transpiled_qc, primitive_backend=backend, shots=shots).result()
        counts = result.quasi_dists[0]
        job_id = result.metadata.get("job_id", None)

        self.job_ids[job_id] = {
            "n_qubits": n_qubits,
            "backend": backend_name,
            "timestamp": start_time
        }
        return job_id, time.time() - start_time, transpiled_qc, counts, backend

    def compare_algorithms(self, max_qubits=5, backend_name=None):
        results = []
        for n in range(2, max_qubits + 1):
            size = 2 ** n
            print(f"\nVerarbeite Größe {size} ...")
            items = list(range(size))
            target = size - 1

            classical_time = np.mean([
                timeit.timeit(lambda: self.linear_search(items, target), number=100)
                for _ in range(3)
            ])

            job_id, quantum_time, _, quantum_counts, backend = self.run_grover(n, backend_name)
            results.append({
                "size": size,
                "classical_time": classical_time,
                "quantum_time": quantum_time,
                "job_id": job_id,
                "n_qubits": n,
                "counts": quantum_counts,
                "backend": backend.name
            })
            print(f"Klassisch: {classical_time:.4f}s | Quanten (Job {job_id}, Backend {backend.name}): {quantum_time:.4f}s")

        self.save_results(results)
        self.plot_results(results)
        return results

    def save_results(self, results, filename="quantum_results.json"):
        import json
        with open(filename, "w") as f:
            json.dump(results, f, indent=2)
        print(f"Ergebnisse gespeichert in {filename}")

    def plot_results(self, results):
        sizes = [r['size'] for r in results]
        classical = [r['classical_time'] for r in results]
        quantum = [r['quantum_time'] for r in results]
        plt.figure(figsize=(10, 6))
        plt.plot(sizes, classical, 'b-o', label='Lineare Suche (O(n))')
        plt.plot(sizes, quantum, 'r--o', label='Grover auf IBMQ (Job-Zeit)')
        plt.xscale('log', base=2)
        plt.yscale('log')
        plt.xlabel('Anzahl der Elemente (log₂-Skala)')
        plt.ylabel('Zeit (Sekunden, log-Skala)')
        plt.title('Leistungsvergleich: Klassisch vs. Quanten (IBM Hardware)')
        plt.legend()
        plt.grid(True, which="both", ls="--")
        plt.savefig('ibmq_grover_results.png', dpi=600, bbox_inches='tight')
        plt.show()


if __name__ == "__main__":
    API_TOKEN = "hc_jCRxZWVqz8KZuZY58kmtbiC81I1tuGuKrU4a-VOGF"          
    INSTANCE = "crn:v1:bluemix:public:quantum-computing:us-east:a/71e0d8f4996f4919a7a1f5a17593eac9:817179d7-c733-47f0-89fa-64f2696e053c::"        
    BACKEND_NAME = "ibm_brisbane"               

    grover = IBMQuantumGrover(api_token=API_TOKEN, instance=INSTANCE)
    results = grover.compare_algorithms(max_qubits=5, backend_name=BACKEND_NAME)


ImportError: Qiskit is installed in an invalid environment that has both Qiskit >=1.0 and an earlier version. You should create a new virtual environment, and ensure that you do not mix dependencies between Qiskit <1.0 and >=1.0. Any packages that depend on 'qiskit-terra' are not compatible with Qiskit 1.0 and will need to be updated. Qiskit unfortunately cannot enforce this requirement during environment resolution. See https://qisk.it/packaging-1-0 for more detail.

In [10]:
service = QiskitRuntimeService(channel="ibm_quantum_platform", token=API_TOKEN, instance='crn:v1:bluemix:public:quantum-computing:us-east:a/71e0d8f4996f4919a7a1f5a17593eac9:817179d7-c733-47f0-89fa-64f2696e053c::')
print([b.name for b in service.backends()])


['ibm_torino', 'ibm_brisbane']


In [13]:
import qiskit
print(qiskit.__version__)

import qiskit_ibm_runtime
print(qiskit_ibm_runtime.__version__)

2.1.1
0.41.0
